# flashinfer_jit_cache smoke test

Built from [FlashInfer's repository](https://github.com/flashinfer-ai/flashinfer) (`flashinfer-jit-cache/` and `flashinfer-jit-cache-provider/`) or installed from FlashInfer's wheel index; not on PyPI.

## Example 1 — the import and the version, which must match FlashInfer's

In [ ]:
import importlib.metadata
import flashinfer_jit_cache

print("flashinfer_jit_cache", flashinfer_jit_cache.__version__, "built from", flashinfer_jit_cache.__git_version__[:10])
assert flashinfer_jit_cache.__version__ == importlib.metadata.version("flashinfer-jit-cache")
# FlashInfer accepts a cache whose version is its own, optionally followed by a +<CUDA> label (for example +cu130).
assert flashinfer_jit_cache.__version__.split("+")[0] == importlib.metadata.version("flashinfer-python").split("+")[0], flashinfer_jit_cache.__version__
assert flashinfer_jit_cache.JIT_CACHE_PROVIDER_ENTRY_POINT_GROUP == "flashinfer.jit_cache.providers"
assert flashinfer_jit_cache.JIT_CACHE_PROVIDER_SCHEMA_VERSION == 1
assert sorted(flashinfer_jit_cache.__all__) == ["JIT_CACHE_PROVIDER_ENTRY_POINT_GROUP", "JIT_CACHE_PROVIDER_SCHEMA_VERSION",
                                                "JitCacheProvider", "get_jit_cache_providers"]
found = flashinfer_jit_cache.get_jit_cache_providers()
assert isinstance(found, tuple) and all(isinstance(p, flashinfer_jit_cache.JitCacheProvider) for p in found)
print(f"OK -- the index imports and scans; {len(found)} kernel pack(s) registered:", [p.provider_id for p in found])

## Example 2 — the scan finds a well-formed provider and ignores a broken one (a stand-in, no real pack)

In [ ]:
import os, sys, tempfile, textwrap
import flashinfer_jit_cache

site = tempfile.mkdtemp(prefix="flashinfer_jit_cache_nb_")  # a tiny stand-in distribution with two entry points
os.makedirs(os.path.join(site, "nb_standin-1.0.dist-info"))
with open(os.path.join(site, "nb_standin-1.0.dist-info", "METADATA"), "w") as handle:
    handle.write("Metadata-Version: 2.1\nName: nb-standin\nVersion: 1.0\n")
with open(os.path.join(site, "nb_standin-1.0.dist-info", "entry_points.txt"), "w") as handle:
    handle.write("[flashinfer.jit_cache.providers]\nnbgood = nb_standin:good\nnbbad = nb_standin:bad\n")
with open(os.path.join(site, "nb_standin.py"), "w") as handle:
    handle.write(textwrap.dedent(f"""
        def good():
            return dict(schema_version=1, distribution="nb-standin", version="1.0", jit_cache_dir={site!r},
                        cuda_architectures=["SM90A", "10.0a"], modules=["norm"])
        def bad():
            return dict(schema_version=99)
    """))
before = {p.provider_id for p in flashinfer_jit_cache.get_jit_cache_providers()}
sys.path.insert(0, site)
try:
    added = {p.provider_id: p for p in flashinfer_jit_cache.get_jit_cache_providers() if p.provider_id not in before}
finally:
    sys.path.remove(site)
assert sorted(added) == ["nbgood"], sorted(added)  # the broken one (unknown schema) is ignored with a warning
# The index normalises every spelling of an architecture to one form (upper case and a dotted version here).
assert added["nbgood"].cuda_architectures == {"sm90a", "sm100a"} and added["nbgood"].modules == {"norm"}, added["nbgood"]
print("OK -- a well-formed provider is found with its architectures normalised; one with an unknown schema is ignored")